Install the required packages:
- jupytergis==0.16.8
- openeo

This requires you restart the server afterwards

In [ ]:
!pip install jupytergis==0.16.8 openeo

In [ ]:
import openeo
from jupytergis import GISDocument
from openeo.processes import process

connection = openeo.connect("https://openeo.mpc.ds.io/")

In [ ]:
connection.authenticate_oidc()

Create graph!

In [ ]:
spatial_extent_east = -73.90
spatial_extent_north = 40.80
spatial_extent_south = 40.70
spatial_extent_west = -74.00

load1 = connection.datacube_from_process(
    "load_collection",
    id="sentinel-2-l2a",
    bands=["B04", "B03", "B02"],
    properties={},
    spatial_extent={
        "east": spatial_extent_east,
        "north": spatial_extent_north,
        "south": spatial_extent_south,
        "west": spatial_extent_west,
    },
    temporal_extent=["2022-04-15T00:00:00Z", "2022-12-31T00:00:00Z"],
)
load1 = load1.reduce_dimension(
    dimension="time",
    reducer="first",
)

def process1(x, context=None):
    data1 = process(
        "linear_scale_range", inputMax=1, inputMin=0, outputMax=255, x=x
    )
    data2 = process("trunc", x=data1)
    return data2

processed = load1.apply(process=process1)

color = processed.process(
    "color_formula",
    data=processed,
    formula="Gamma RGB 1.5 Sigmoidal RGB 6 0.3 Saturation 1",
)

graph = color

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=40.7128, longitude=-74.0060, zoom=12)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="© OpenStreetMap contributors",
)


This will create the tile service and show the layer dynamically

In [ ]:
doc.add_openeo_tile_layer(graph)
doc